# Multimodal RAG for Chest X-ray Report Generation

## What this notebook does

This notebook implements the **retrieval-augmented generation (RAG)** layer of our chest X-ray report generation system. The pipeline:

1. Takes the multi-label predictions from our ViT-B/16 classifier (already trained on NIH ChestX-ray14, ~0.71 mean AUC).
2. Retrieves the most relevant snippets from a curated **radiology guideline knowledge base**.
3. Feeds predicted labels + retrieved snippets to an LLM, which produces a structured report with `Findings` and `Impression` sections.
4. Runs a lightweight **verifier** that checks consistency between predicted labels and the generated text (the "reduces hallucination" claim from our problem statement).

## Architecture

```
CXR image
   │
   ▼
ViT-B/16 ──► predicted labels (e.g., {Effusion: 0.81, Infiltration: 0.42})
                    │
                    ▼
        ┌──── Hybrid Retriever ────┐
        │  - Dense (PubMedBERT)    │
        │  - BM25 (lexical)        │
        │  - Label-conditioned     │
        │  - Reciprocal Rank Fusion│
        └──────────────────────────┘
                    │
                    ▼
            top-N guideline snippets
                    │
                    ▼
         LLM (prompt with labels + snippets)
                    │
                    ▼
         Findings → Impression (two-stage)
                    │
                    ▼
              Verifier (consistency check)
```

**Run order:** execute cells top-to-bottom. The first run will download the embedding model (~440 MB).

## 1. Setup

Install dependencies. Use a GPU runtime in Colab (`Runtime → Change runtime type → T4 GPU`) — embeddings are much faster on GPU.

In [9]:


!pip install -q -U "sentence-transformers" "faiss-cpu>=1.9.0" "rank-bm25"
!pip install numpy
# Optional generator backends — uncomment what you plan to use:
# !pip install -q openai
# !pip install -q anthropic


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 117.8 MB/s eta 0:00:00


In [19]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
import json
import re
import pickle
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np

print("Imports OK")

Imports OK


## 2. Knowledge Base

This is the heart of the RAG system. For each of the 14 NIH ChestX-ray14 pathologies we curate **3–4 guideline-style snippets** covering:

- **`definition`** — what the condition is.
- **`feature`** — how it appears on a chest radiograph.
- **`phrasing`** — example sentences a radiologist would write in a report.
- **`pitfall`** — common mistakes / differential considerations.

The snippets are sourced from standard radiology references (Radiopaedia, Fleischner Society guidelines, ChestX-ray8 paper, StatPearls). To extend the KB, just append more dicts — the index will pick them up on rebuild.

> **Why this matters:** the quality of these snippets directly determines the quality of the generated reports. Garbage in, garbage out. Spend time here.

In [3]:
KNOWLEDGE_BASE = [
    # ---------------- Atelectasis ----------------
    {"id": "atelectasis_def", "pathology": "Atelectasis", "type": "definition", "source": "Radiopaedia",
     "text": "Atelectasis refers to collapse or incomplete expansion of lung tissue, leading to reduced gas exchange in the affected region. It can be obstructive (mucus plug, tumor, foreign body), compressive (mass or effusion), or passive (post-operative, hypoventilation)."},
    {"id": "atelectasis_feat1", "pathology": "Atelectasis", "type": "feature", "source": "Radiopaedia",
     "text": "On chest radiographs, atelectasis appears as increased opacity with associated VOLUME LOSS. Indirect signs include displacement of fissures, ipsilateral mediastinal shift, elevation of the hemidiaphragm, and crowding of bronchovascular markings."},
    {"id": "atelectasis_feat2", "pathology": "Atelectasis", "type": "feature", "source": "Radiopaedia",
     "text": "Subsegmental (linear, discoid, or plate-like) atelectasis presents as thin linear opacities, most commonly in the lung bases, often related to poor inspiration or post-operative states."},
    {"id": "atelectasis_phrase", "pathology": "Atelectasis", "type": "phrasing", "source": "ACR style guide",
     "text": "Example phrasing: 'Linear opacity in the right lower lobe consistent with subsegmental atelectasis.' or 'Volume loss in the left lower lobe with associated opacity, consistent with atelectasis.'"},

    # ---------------- Cardiomegaly ----------------
    {"id": "cardiomegaly_def", "pathology": "Cardiomegaly", "type": "definition", "source": "Radiopaedia",
     "text": "Cardiomegaly is enlargement of the cardiac silhouette beyond normal limits. Causes include chamber dilation, hypertrophy, pericardial effusion, or technical factors such as AP projection."},
    {"id": "cardiomegaly_feat", "pathology": "Cardiomegaly", "type": "feature", "source": "Radiopaedia",
     "text": "On a PA chest radiograph, cardiomegaly is defined as a cardiothoracic ratio (CTR) greater than 0.5, where CTR = maximum transverse cardiac diameter / maximum transverse thoracic diameter at the level of the diaphragm."},
    {"id": "cardiomegaly_phrase", "pathology": "Cardiomegaly", "type": "phrasing", "source": "ACR style guide",
     "text": "Example phrasing: 'Enlarged cardiac silhouette with cardiothoracic ratio greater than 0.5, consistent with cardiomegaly.'"},
    {"id": "cardiomegaly_pitfall", "pathology": "Cardiomegaly", "type": "pitfall", "source": "Radiopaedia",
     "text": "Pitfall: AP projection magnifies the cardiac silhouette by 15-20%; do not over-call cardiomegaly on AP supine films, especially in ICU portables."},

    # ---------------- Consolidation ----------------
    {"id": "consolidation_def", "pathology": "Consolidation", "type": "definition", "source": "Fleischner Society",
     "text": "Consolidation refers to replacement of alveolar air with fluid, pus, blood, or cells, producing a region of increased opacity that obscures the underlying vessels and airways."},
    {"id": "consolidation_feat", "pathology": "Consolidation", "type": "feature", "source": "Fleischner Society",
     "text": "Radiographic features include homogeneous opacity, AIR BRONCHOGRAMS (lucent branching airways within the opacity), the SILHOUETTE SIGN (loss of normal anatomic borders adjacent to the opacity), and absence of significant volume loss."},
    {"id": "consolidation_phrase", "pathology": "Consolidation", "type": "phrasing", "source": "ACR style guide",
     "text": "Example phrasing: 'Air-space opacity in the right lower lobe with air bronchograms, consistent with consolidation. Differential includes pneumonia, aspiration, and pulmonary edema.'"},

    # ---------------- Edema ----------------
    {"id": "edema_def", "pathology": "Edema", "type": "definition", "source": "Radiopaedia",
     "text": "Pulmonary edema is the abnormal accumulation of fluid in the extravascular spaces of the lung, most commonly from elevated pulmonary venous pressure (cardiogenic) or increased capillary permeability (non-cardiogenic / ARDS)."},
    {"id": "edema_feat", "pathology": "Edema", "type": "feature", "source": "Radiopaedia",
     "text": "Cardiogenic edema progresses through stages: cephalization of pulmonary vasculature, peribronchial cuffing, Kerley B lines, perihilar 'bat-wing' haze, and frank alveolar edema. Pleural effusions are common."},
    {"id": "edema_phrase", "pathology": "Edema", "type": "phrasing", "source": "ACR style guide",
     "text": "Example phrasing: 'Bilateral perihilar opacities with Kerley B lines and small pleural effusions, consistent with pulmonary edema in the setting of cardiac decompensation.'"},

    # ---------------- Effusion ----------------
    {"id": "effusion_def", "pathology": "Effusion", "type": "definition", "source": "Radiopaedia",
     "text": "Pleural effusion is the abnormal accumulation of fluid in the pleural space. Causes are broadly classified as transudative (heart failure, cirrhosis) or exudative (infection, malignancy, inflammation)."},
    {"id": "effusion_feat1", "pathology": "Effusion", "type": "feature", "source": "Radiopaedia",
     "text": "On upright PA radiographs, effusions blunt the costophrenic angle (visible at >= 200 mL) and form a meniscus along the lateral chest wall. Lateral views may detect smaller effusions (>= 50 mL) as blunting of the posterior costophrenic angle."},
    {"id": "effusion_feat2", "pathology": "Effusion", "type": "feature", "source": "Radiopaedia",
     "text": "Large effusions can opacify the entire hemithorax with contralateral mediastinal shift. On supine films, effusions layer posteriorly producing a diffuse veiling opacity without clearly obscuring the costophrenic angle."},
    {"id": "effusion_phrase", "pathology": "Effusion", "type": "phrasing", "source": "ACR style guide",
     "text": "Example phrasing: 'Blunting of the right costophrenic angle with meniscus, consistent with small to moderate right pleural effusion.'"},

    # ---------------- Emphysema ----------------
    {"id": "emphysema_def", "pathology": "Emphysema", "type": "definition", "source": "Radiopaedia",
     "text": "Emphysema is permanent enlargement of the air spaces distal to the terminal bronchioles with destruction of alveolar walls; a major component of chronic obstructive pulmonary disease (COPD)."},
    {"id": "emphysema_feat", "pathology": "Emphysema", "type": "feature", "source": "Radiopaedia",
     "text": "Radiographic features include hyperinflated lungs with flattened hemidiaphragms, increased retrosternal clear space on lateral view, bullae (thin-walled radiolucent areas), narrow vertical cardiac silhouette, and decreased peripheral vascular markings."},
    {"id": "emphysema_phrase", "pathology": "Emphysema", "type": "phrasing", "source": "ACR style guide",
     "text": "Example phrasing: 'Hyperinflated lungs with flattened diaphragms and increased retrosternal clear space, consistent with emphysema/COPD.'"},

    # ---------------- Fibrosis ----------------
    {"id": "fibrosis_def", "pathology": "Fibrosis", "type": "definition", "source": "Radiopaedia",
     "text": "Pulmonary fibrosis is scarring of the lung interstitium leading to architectural distortion and impaired gas exchange. Causes include idiopathic pulmonary fibrosis (IPF), connective tissue disease, drug toxicity, and chronic hypersensitivity pneumonitis."},
    {"id": "fibrosis_feat", "pathology": "Fibrosis", "type": "feature", "source": "Radiopaedia",
     "text": "Radiographic features include reticular (net-like) opacities, often basal and peripheral in IPF, with associated volume loss and traction bronchiectasis. Honeycombing (clustered cystic spaces) indicates end-stage fibrosis and is best seen on HRCT."},
    {"id": "fibrosis_phrase", "pathology": "Fibrosis", "type": "phrasing", "source": "ACR style guide",
     "text": "Example phrasing: 'Bilateral reticular opacities with peripheral and basal predominance and associated volume loss, consistent with fibrotic lung disease.'"},

    # ---------------- Hernia ----------------
    {"id": "hernia_def", "pathology": "Hernia", "type": "definition", "source": "Radiopaedia",
     "text": "A diaphragmatic hernia is the protrusion of abdominal contents into the thoracic cavity through a defect in the diaphragm. The most common is a hiatal hernia, where the gastric cardia herniates through the esophageal hiatus."},
    {"id": "hernia_feat", "pathology": "Hernia", "type": "feature", "source": "Radiopaedia",
     "text": "A retrocardiac air-fluid level is the classic chest radiograph finding of a hiatal hernia. Larger hernias may show a soft-tissue mass behind the heart with or without air content."},
    {"id": "hernia_phrase", "pathology": "Hernia", "type": "phrasing", "source": "ACR style guide",
     "text": "Example phrasing: 'Retrocardiac air-fluid level consistent with hiatal hernia.'"},

    # ---------------- Infiltration ----------------
    {"id": "infiltration_def", "pathology": "Infiltration", "type": "definition", "source": "Fleischner Society",
     "text": "'Infiltration' is a non-specific radiographic term describing abnormal opacities in the lung parenchyma that may be alveolar (air-space) or interstitial. Modern radiology guidelines prefer more specific terminology such as 'consolidation', 'ground-glass opacity', or 'interstitial opacity'."},
    {"id": "infiltration_feat", "pathology": "Infiltration", "type": "feature", "source": "Fleischner Society",
     "text": "Air-space (alveolar) infiltration produces fluffy, ill-defined opacities with possible air bronchograms. Interstitial infiltration produces reticular, nodular, or reticulonodular patterns."},
    {"id": "infiltration_phrase", "pathology": "Infiltration", "type": "phrasing", "source": "ACR style guide",
     "text": "Example phrasing: 'Patchy air-space opacity in the right middle lobe, suggestive of infiltration; differential includes pneumonia, aspiration, or atelectasis.'"},

    # ---------------- Mass ----------------
    {"id": "mass_def", "pathology": "Mass", "type": "definition", "source": "Fleischner Society",
     "text": "A pulmonary mass is a focal opacity in the lung parenchyma measuring greater than 3 cm. Lesions <= 3 cm are termed 'nodules'."},
    {"id": "mass_feat", "pathology": "Mass", "type": "feature", "source": "Fleischner Society",
     "text": "Features suggestive of malignancy include spiculated margins, lobulated contour, large size, growth on serial imaging, and associated lymphadenopathy. Cavitation may be present."},
    {"id": "mass_phrase", "pathology": "Mass", "type": "phrasing", "source": "ACR style guide",
     "text": "Example phrasing: '4 cm mass in the right upper lobe with spiculated margins; malignancy cannot be excluded. Recommend further evaluation with chest CT.'"},

    # ---------------- Nodule ----------------
    {"id": "nodule_def", "pathology": "Nodule", "type": "definition", "source": "Fleischner Society",
     "text": "A pulmonary nodule is a rounded or irregular opacity in the lung parenchyma measuring up to 3 cm in greatest dimension. Nodules <= 8 mm are sometimes called sub-centimeter or small nodules."},
    {"id": "nodule_feat", "pathology": "Nodule", "type": "feature", "source": "Fleischner Society",
     "text": "Benign features include central or popcorn-like calcification (granuloma, hamartoma), small size, and stability for >= 2 years. Suspicious features include spiculation, large size (> 8 mm), upper-lobe location, and growth."},
    {"id": "nodule_phrase", "pathology": "Nodule", "type": "phrasing", "source": "Fleischner Society 2017",
     "text": "Example phrasing: '6 mm nodule in the right upper lobe; recommend correlation with prior imaging or follow-up per Fleischner Society guidelines.'"},
    {"id": "nodule_pitfall", "pathology": "Nodule", "type": "pitfall", "source": "Radiopaedia",
     "text": "Pitfall: solitary pulmonary nodules are often missed on chest radiograph, especially when projected over ribs, clavicles, or hila. Chest CT is significantly more sensitive."},

    # ---------------- Pleural Thickening ----------------
    {"id": "plthick_def", "pathology": "Pleural_Thickening", "type": "definition", "source": "Radiopaedia",
     "text": "Pleural thickening is fibrous thickening of the pleural surfaces, focal or diffuse. Causes include prior infection (TB, empyema), hemothorax, asbestos exposure, and malignancy (mesothelioma)."},
    {"id": "plthick_feat", "pathology": "Pleural_Thickening", "type": "feature", "source": "Radiopaedia",
     "text": "Radiographic features include irregular thickening of the pleural surface, blunting of the costophrenic angle WITHOUT a fluid meniscus, and (if calcified) plaques along the diaphragm and posterolateral chest wall — the latter strongly suggesting asbestos exposure."},
    {"id": "plthick_phrase", "pathology": "Pleural_Thickening", "type": "phrasing", "source": "ACR style guide",
     "text": "Example phrasing: 'Focal pleural thickening along the right lateral chest wall, possibly post-inflammatory; correlation with prior imaging recommended.'"},

    # ---------------- Pneumonia ----------------
    {"id": "pneumonia_def", "pathology": "Pneumonia", "type": "definition", "source": "Radiopaedia",
     "text": "Pneumonia is acute infection of the lung parenchyma causing inflammation and consolidation, typically due to bacterial, viral, or fungal pathogens."},
    {"id": "pneumonia_feat", "pathology": "Pneumonia", "type": "feature", "source": "Radiopaedia",
     "text": "Radiographic patterns include lobar consolidation (classic Streptococcus pneumoniae), bronchopneumonia (patchy multifocal opacities), and interstitial pneumonia (reticular opacities, often viral or atypical). Air bronchograms are common in lobar pneumonia."},
    {"id": "pneumonia_phrase", "pathology": "Pneumonia", "type": "phrasing", "source": "ACR style guide",
     "text": "Example phrasing: 'Air-space opacity in the right lower lobe with air bronchograms, consistent with pneumonia in the appropriate clinical setting.'"},
    {"id": "pneumonia_pitfall", "pathology": "Pneumonia", "type": "pitfall", "source": "Radiopaedia",
     "text": "Pitfall: radiographic findings may lag clinical symptoms by 24-48 hours and may persist for weeks after clinical resolution. Follow-up imaging is recommended for resolution, particularly in older patients, to exclude underlying malignancy."},

    # ---------------- Pneumothorax ----------------
    {"id": "ptx_def", "pathology": "Pneumothorax", "type": "definition", "source": "Radiopaedia",
     "text": "Pneumothorax is the abnormal presence of air in the pleural space, causing partial or complete lung collapse. Tension pneumothorax is a life-threatening emergency."},
    {"id": "ptx_feat1", "pathology": "Pneumothorax", "type": "feature", "source": "Radiopaedia",
     "text": "The hallmark finding is a visible visceral pleural line displaced from the chest wall with absence of lung markings peripheral to it. Best seen on upright expiratory radiographs at the lung apex."},
    {"id": "ptx_feat2", "pathology": "Pneumothorax", "type": "feature", "source": "Radiopaedia",
     "text": "Tension pneumothorax shows contralateral mediastinal shift, ipsilateral hemidiaphragm depression, and patient hemodynamic instability — requires immediate decompression."},
    {"id": "ptx_phrase", "pathology": "Pneumothorax", "type": "phrasing", "source": "ACR style guide",
     "text": "Example phrasing: 'Visible visceral pleural line in the right apex with absence of distal lung markings, consistent with small right pneumothorax.'"},
    {"id": "ptx_pitfall", "pathology": "Pneumothorax", "type": "pitfall", "source": "Radiopaedia",
     "text": "Pitfall: on supine films, pneumothorax may collect anteriorly producing a 'deep sulcus sign' (abnormally deep, lucent costophrenic angle) rather than an apical pleural line. Skin folds can mimic a pneumothorax line; trace any suspected line beyond the chest wall to disprove."},
]

PATHOLOGIES = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Effusion",
    "Emphysema", "Fibrosis", "Hernia", "Infiltration", "Mass",
    "Nodule", "Pleural_Thickening", "Pneumonia", "Pneumothorax",
]

print(f"Knowledge base loaded: {len(KNOWLEDGE_BASE)} snippets across {len(PATHOLOGIES)} pathologies")

Knowledge base loaded: 49 snippets across 14 pathologies


## 3. Build the FAISS Index

We embed every snippet with **PubMedBERT** (a medical-domain sentence encoder) and store the vectors in a FAISS flat index. The KB is small (~50 snippets) so a flat index is plenty fast.

> **Why PubMedBERT?** It was pretrained on PubMed abstracts, so it understands radiology vocabulary far better than generic embedders like `text-embedding-ada-002`. If you don't want a medical model, swap in `BAAI/bge-small-en-v1.5` — it's lighter and still solid.

The first run downloads the model (~440 MB). Subsequent runs use the cached copy.

In [10]:
from sentence_transformers import SentenceTransformer
import faiss

# Medical-domain (recommended). Fallback: "BAAI/bge-small-en-v1.5"
EMBED_MODEL_NAME = "pritamdeka/S-PubMedBert-MS-MARCO"

print(f"Loading embedder: {EMBED_MODEL_NAME}")
embedder = SentenceTransformer(EMBED_MODEL_NAME)
print(f"Embedding dim: {embedder.get_sentence_embedding_dimension()}")

Loading embedder: pritamdeka/S-PubMedBert-MS-MARCO


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dim: 768


In [11]:
# Embed all snippets
texts = [s["text"] for s in KNOWLEDGE_BASE]
embeddings = embedder.encode(
    texts,
    normalize_embeddings=True,    # so inner-product == cosine similarity
    show_progress_bar=True,
    batch_size=16,
).astype("float32")

# Build flat IP (cosine) index
faiss_index = faiss.IndexFlatIP(embeddings.shape[1])
faiss_index.add(embeddings)

print(f"FAISS index built: {faiss_index.ntotal} vectors of dim {embeddings.shape[1]}")

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

FAISS index built: 49 vectors of dim 768


## 4. Hybrid Retriever

Our retriever combines three signals:

1. **Dense semantic search** — FAISS over PubMedBERT embeddings. Catches paraphrases.
2. **BM25 lexical search** — token-level matching. Catches exact terminology.
3. **Label-conditioned filter** — boost snippets whose `pathology` tag matches a predicted label. This is the killer feature: when the classifier is confident about *Effusion*, we **guarantee** Effusion snippets appear in context.

All three rankings are merged via **Reciprocal Rank Fusion (RRF)** — a parameter-free fusion method that consistently beats individual rankers.

The retriever's public API is one method:

```python
retriever.retrieve(predicted_labels: dict[str, float]) -> list[Snippet]
```

`predicted_labels` is whatever the ViT outputs (a dict mapping pathology name → probability). The retriever filters to confident labels (default threshold 0.3), runs per-label retrieval, and aggregates.

In [12]:
from rank_bm25 import BM25Okapi

def _tokenize(text: str) -> List[str]:
    return re.findall(r"[a-z0-9]+", text.lower())

# Pre-tokenize the corpus once for BM25
_tokenized_corpus = [_tokenize(s["text"]) for s in KNOWLEDGE_BASE]
bm25 = BM25Okapi(_tokenized_corpus)
print("BM25 ready")

BM25 ready


In [13]:
class Retriever:
    def __init__(self,
                 label_threshold: float = 0.3,
                 top_n_total: int = 8,
                 per_label_k: int = 5,
                 rrf_k: int = 60):
        """
        label_threshold: minimum probability for a label to trigger retrieval
        top_n_total:     max number of snippets to return overall
        per_label_k:     snippets retrieved per active label before fusion
        rrf_k:           RRF smoothing constant (60 is the standard value)
        """
        self.label_threshold = label_threshold
        self.top_n_total = top_n_total
        self.per_label_k = per_label_k
        self.rrf_k = rrf_k

    # ---- individual retrieval signals ----

    def _dense(self, query: str, k: int) -> List[Tuple[int, float]]:
        emb = embedder.encode([query], normalize_embeddings=True).astype("float32")
        scores, idxs = faiss_index.search(emb, k)
        return list(zip(idxs[0].tolist(), scores[0].tolist()))

    def _bm25(self, query: str, k: int) -> List[Tuple[int, float]]:
        scores = bm25.get_scores(_tokenize(query))
        top = np.argsort(scores)[::-1][:k]
        return [(int(i), float(scores[i])) for i in top]

    def _label_match(self, label: str) -> List[Tuple[int, float]]:
        norm = label.lower().replace("_", " ")
        return [
            (i, 1.0) for i, s in enumerate(KNOWLEDGE_BASE)
            if s["pathology"].lower().replace("_", " ") == norm
        ]

    # ---- Reciprocal Rank Fusion ----

    def _rrf(self, ranked_lists: List[List[Tuple[int, float]]]) -> List[Tuple[int, float]]:
        scores: Dict[int, float] = {}
        for ranked in ranked_lists:
            for rank, (idx, _) in enumerate(ranked):
                scores[idx] = scores.get(idx, 0.0) + 1.0 / (self.rrf_k + rank + 1)
        return sorted(scores.items(), key=lambda x: -x[1])

    # ---- public API ----

    def retrieve(self, predicted_labels: Dict[str, float]) -> List[dict]:
        # 1. pick active labels
        active = [(lbl, p) for lbl, p in predicted_labels.items()
                  if p >= self.label_threshold]
        if not active:
            # fall back to top-1 even if below threshold, so we always return SOMETHING
            active = [max(predicted_labels.items(), key=lambda x: x[1])]

        # 2. per-label hybrid retrieval, fused via RRF
        agg: Dict[int, float] = {}
        for label, prob in active:
            query = f"{label} chest x-ray imaging features findings reporting"
            dense = self._dense(query, k=self.per_label_k * 2)
            lex   = self._bm25(query, k=self.per_label_k * 2)
            match = self._label_match(label)
            fused = self._rrf([dense, lex, match])[:self.per_label_k]
            for idx, score in fused:
                # weight by label probability so high-confidence labels dominate
                agg[idx] = max(agg.get(idx, 0.0), score * prob)

        # 3. global ranking
        ranked = sorted(agg.items(), key=lambda x: -x[1])[:self.top_n_total]
        results = []
        for idx, score in ranked:
            s = dict(KNOWLEDGE_BASE[idx])
            s["score"] = float(score)
            results.append(s)
        return results

retriever = Retriever()
print("Retriever ready")

Retriever ready


## 5. Generator

Now we feed the predicted labels + retrieved snippets into an LLM. The generator supports four backends:

| Backend     | When to use it                                                          |
|-------------|-------------------------------------------------------------------------|
| `dryrun`    | Default. Prints the full prompt without calling any model. Use for debugging your retrieval. |
| `openai`    | GPT-4o or GPT-4o-mini. Set `OPENAI_API_KEY`.                           |
| `anthropic` | Claude. Set `ANTHROPIC_API_KEY`.                                       |
| `hf`        | Local HuggingFace model (e.g., Llama-3-8B-Instruct). Needs GPU.        |

The prompt enforces our two-stage workflow: **Findings first, then Impression**, both grounded in the retrieved snippets and the predicted labels. Hedging language is encouraged when label probabilities are low.

In [14]:
SYSTEM_PROMPT = """You are an experienced radiologist drafting a chest X-ray report.

Generate the report in two sections:

1. FINDINGS: Describe the imaging findings systematically (lungs, heart, mediastinum, pleura, bones, soft tissues). Reference only the predicted abnormalities and the provided guideline context. Do NOT invent findings not supported by either.

2. IMPRESSION: Provide a concise clinical conclusion based strictly on the Findings.

Use formal radiology phrasing. If a predicted abnormality has low probability, use hedging language ("possible", "cannot be excluded", "suggestive of"). If the predicted labels indicate a normal study, state that clearly."""


def build_user_prompt(predicted_labels: Dict[str, float], snippets: List[dict]) -> str:
    label_lines = "\n".join(
        f"  - {lbl}: {p:.2f}"
        for lbl, p in sorted(predicted_labels.items(), key=lambda x: -x[1])
        if p >= 0.2
    ) or "  (no labels above 0.20 — likely a normal study)"

    snippet_blocks = "\n\n".join(
        f"[{s['pathology']} | {s['type']} | source: {s['source']}]\n{s['text']}"
        for s in snippets
    )

    return f"""PREDICTED ABNORMALITIES (from vision model, with probabilities):
{label_lines}

RETRIEVED RADIOLOGY GUIDELINE CONTEXT:
{snippet_blocks}

Generate the FINDINGS section, then the IMPRESSION section, of the radiology report."""

In [15]:
class Generator:
    def __init__(self, backend: str = "dryrun", model: Optional[str] = None,
                 api_key: Optional[str] = None, temperature: float = 0.2):
        self.backend = backend
        self.temperature = temperature
        self.model = model

        if backend == "openai":
            from openai import OpenAI
            import os
            self.client = OpenAI(api_key=api_key or os.getenv("OPENAI_API_KEY"))
            self.model = model or "gpt-4o-mini"

        elif backend == "anthropic":
            import anthropic, os
            self.client = anthropic.Anthropic(api_key=api_key or os.getenv("ANTHROPIC_API_KEY"))
            self.model = model or "claude-sonnet-4-5"

        elif backend == "hf":
            from transformers import pipeline
            self.client = pipeline(
                "text-generation",
                model=model or "meta-llama/Meta-Llama-3-8B-Instruct",
                device_map="auto",
            )

        elif backend == "dryrun":
            self.client = None

        else:
            raise ValueError(f"Unknown backend: {backend}")

    def generate(self, predicted_labels: Dict[str, float], snippets: List[dict]) -> str:
        user_prompt = build_user_prompt(predicted_labels, snippets)

        if self.backend == "dryrun":
            return (
                "=== SYSTEM PROMPT ===\n" + SYSTEM_PROMPT
                + "\n\n=== USER PROMPT ===\n" + user_prompt
                + "\n\n=== [DRYRUN — would call LLM here] ==="
            )

        if self.backend == "openai":
            resp = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=self.temperature,
            )
            return resp.choices[0].message.content

        if self.backend == "anthropic":
            resp = self.client.messages.create(
                model=self.model,
                max_tokens=1024,
                system=SYSTEM_PROMPT,
                messages=[{"role": "user", "content": user_prompt}],
            )
            return resp.content[0].text

        if self.backend == "hf":
            full = SYSTEM_PROMPT + "\n\n" + user_prompt
            out = self.client(full, max_new_tokens=512, temperature=self.temperature,
                              do_sample=self.temperature > 0)
            return out[0]["generated_text"][len(full):]

generator = Generator(backend="dryrun")
print("Generator ready (dryrun mode)")

Generator ready (dryrun mode)


## 6. Verifier (Hallucination Check)

The verifier is a lightweight consistency checker between the predicted labels and the conditions actually mentioned in the generated report. It produces three numbers we can report:

- **Hallucinated** — conditions in the report that the classifier did NOT predict (potentially fabricated).
- **Missed** — conditions the classifier predicted with high confidence but the report didn't mention.
- **Consistency rate** — fraction of generated reports with zero hallucinated mentions.

This is a simple regex-based labeler. For your final paper you could swap in [CheXbert](https://github.com/stanfordmlgroup/CheXbert) — same idea, much more accurate.

In [16]:
SYNONYMS = {
    "Atelectasis":         [r"atelectasis", r"\bcollapse\b"],
    "Cardiomegaly":        [r"cardiomegaly", r"enlarged heart", r"cardiac enlargement", r"cardiothoracic ratio"],
    "Consolidation":       [r"consolidation"],
    "Edema":               [r"\bedema\b", r"pulmonary edema", r"kerley"],
    "Effusion":            [r"effusion", r"pleural fluid"],
    "Emphysema":           [r"emphysema", r"hyperinflation", r"hyperinflated"],
    "Fibrosis":            [r"fibrosis", r"fibrotic", r"honeycomb"],
    "Hernia":              [r"hernia"],
    "Infiltration":        [r"infiltrat"],
    "Mass":                [r"\bmass\b"],
    "Nodule":              [r"nodule"],
    "Pleural_Thickening":  [r"pleural thickening"],
    "Pneumonia":           [r"pneumonia"],
    "Pneumothorax":        [r"pneumothorax"],
}

def verify(report: str, predicted_labels: Dict[str, float],
           confidence_threshold: float = 0.4) -> dict:
    text = report.lower()
    confident = {l for l, p in predicted_labels.items() if p >= confidence_threshold}

    mentioned = set()
    for label, patterns in SYNONYMS.items():
        if any(re.search(p, text) for p in patterns):
            mentioned.add(label)

    hallucinated = mentioned - confident
    missed = confident - mentioned
    return {
        "mentioned":          sorted(mentioned),
        "predicted_confident": sorted(confident),
        "hallucinated":       sorted(hallucinated),
        "missed":             sorted(missed),
        "is_consistent":      len(hallucinated) == 0 and len(missed) == 0,
    }

print("Verifier ready")

Verifier ready


## 7. End-to-End Pipeline

Glue everything together. One call: predicted labels → retrieved snippets → generated report → verifier output.

In [17]:
def run_pipeline(predicted_labels: Dict[str, float],
                 retriever: Retriever,
                 generator: Generator) -> dict:
    snippets = retriever.retrieve(predicted_labels)
    report = generator.generate(predicted_labels, snippets)
    # Verifier only meaningful for non-dryrun outputs, but we always run it
    verifier_out = verify(report, predicted_labels)
    return {
        "predicted_labels":   predicted_labels,
        "retrieved_snippets": snippets,
        "report":             report,
        "verifier":           verifier_out,
    }

print("Pipeline ready")

Pipeline ready


## 8. Demo — Run on a Mock Classifier Output

This simulates what happens at inference time. Replace `mock_predicted_labels` with the real ViT output once you wire it up (see next section).

The example below mimics a patient with a moderate right pleural effusion plus some Infiltration suspicion.

In [18]:
mock_predicted_labels = {
    "Atelectasis":        0.12,
    "Cardiomegaly":       0.18,
    "Consolidation":      0.07,
    "Edema":              0.05,
    "Effusion":           0.81,
    "Emphysema":          0.09,
    "Fibrosis":           0.04,
    "Hernia":             0.02,
    "Infiltration":       0.42,
    "Mass":               0.08,
    "Nodule":             0.11,
    "Pleural_Thickening": 0.15,
    "Pneumonia":          0.06,
    "Pneumothorax":       0.03,
}

result = run_pipeline(mock_predicted_labels, retriever, generator)

print("=" * 72)
print("TOP-5 PREDICTED LABELS (ViT)")
print("=" * 72)
for lbl, p in sorted(mock_predicted_labels.items(), key=lambda x: -x[1])[:5]:
    print(f"  {lbl:25s}  {p:.3f}")

print("\n" + "=" * 72)
print(f"RETRIEVED SNIPPETS  (n={len(result['retrieved_snippets'])})")
print("=" * 72)
for i, s in enumerate(result["retrieved_snippets"], 1):
    print(f"\n[{i}] {s['pathology']:20s} {s['type']:12s} score={s['score']:.4f}  src={s['source']}")
    txt = s["text"]
    print(f"    {txt[:220]}{'...' if len(txt) > 220 else ''}")

print("\n" + "=" * 72)
print("GENERATED REPORT")
print("=" * 72)
print(result["report"])

print("\n" + "=" * 72)
print("VERIFIER")
print("=" * 72)
print(json.dumps(result["verifier"], indent=2))

TOP-5 PREDICTED LABELS (ViT)
  Effusion                   0.810
  Infiltration               0.420
  Cardiomegaly               0.180
  Pleural_Thickening         0.150
  Atelectasis                0.120

RETRIEVED SNIPPETS  (n=8)

[1] Effusion             definition   score=0.0368  src=Radiopaedia
    Pleural effusion is the abnormal accumulation of fluid in the pleural space. Causes are broadly classified as transudative (heart failure, cirrhosis) or exudative (infection, malignancy, inflammation).

[2] Pleural_Thickening   feature      score=0.0255  src=Radiopaedia
    Radiographic features include irregular thickening of the pleural surface, blunting of the costophrenic angle WITHOUT a fluid meniscus, and (if calcified) plaques along the diaphragm and posterolateral chest wall — the l...

[3] Effusion             phrasing     score=0.0251  src=ACR style guide
    Example phrasing: 'Blunting of the right costophrenic angle with meniscus, consistent with small to moderate right pleur

In [25]:
!pip install -q timm

In [26]:
import torch
import torch.nn as nn
import timm
from torchvision import transforms
from PIL import Image

class ViTWrapper(nn.Module):
    def __init__(self, num_classes=14):
        super().__init__()
        self.model = timm.create_model(
            "vit_base_patch16_224",
            pretrained=False,
            num_classes=num_classes,   # timm replaces the head internally
        )

    def forward(self, x):
        return self.model(x)


def load_vit_b16_with_14_class_head(checkpoint_path: str, device: str = "cuda") -> nn.Module:
    model = ViTWrapper(num_classes=14)

    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    if isinstance(ckpt, nn.Module):
        return ckpt.to(device).eval()
    if isinstance(ckpt, dict):
        for key in ("model_state_dict", "state_dict"):
            if key in ckpt and isinstance(ckpt[key], dict):
                ckpt = ckpt[key]
                break

    missing, unexpected = model.load_state_dict(ckpt, strict=False)
    print(f"Missing: {len(missing)}, Unexpected: {len(unexpected)}")
    if missing:    print("  first missing:", missing[:3])
    if unexpected: print("  first unexpected:", unexpected[:3])

    return model.to(device).eval()


preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

PATHOLOGIES = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Effusion",
    "Emphysema", "Fibrosis", "Hernia", "Infiltration", "Mass",
    "Nodule", "Pleural_Thickening", "Pneumonia", "Pneumothorax",
]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
vit = load_vit_b16_with_14_class_head(
    "/content/drive/MyDrive/Data/Model/vit_base_nih_results_01.pth",
    device=DEVICE,
)

Missing: 0, Unexpected: 0


In [27]:
import torch
import torch.nn as nn
from torchvision.models import vit_b_16
from torchvision import transforms
from PIL import Image
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
PATHOLOGIES = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Effusion",
    "Emphysema", "Fibrosis", "Hernia", "Infiltration", "Mass",
    "Nodule", "Pleural_Thickening", "Pneumonia", "Pneumothorax",
]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
vit = load_vit_b16_with_14_class_head(
    "/content/drive/MyDrive/Data/Model/vit_base_nih_results_01.pth",
    device=DEVICE,
)

def predict_labels_for_image(image_path: str) -> dict:
    img = Image.open(image_path).convert("RGB")
    x = preprocess(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = vit(x)
        probs = torch.sigmoid(logits)[0].cpu().numpy()
    return dict(zip(PATHOLOGIES, probs.tolist()))
preds = predict_labels_for_image("/content/drive/MyDrive/Data/images_001/images/00000001_000.png")  # change to a real image path
result = run_pipeline(preds, retriever, generator)

print("Top-5 predicted:")
for lbl, p in sorted(preds.items(), key=lambda x: -x[1])[:5]:
    print(f"  {lbl:25s} {p:.3f}")
print("\n--- Generated report ---")
print(result["report"])

Missing: 0, Unexpected: 0
Top-5 predicted:
  Infiltration              0.273
  Atelectasis               0.253
  Effusion                  0.209
  Pleural_Thickening        0.159
  Cardiomegaly              0.141

--- Generated report ---
=== SYSTEM PROMPT ===
You are an experienced radiologist drafting a chest X-ray report.

Generate the report in two sections:

1. FINDINGS: Describe the imaging findings systematically (lungs, heart, mediastinum, pleura, bones, soft tissues). Reference only the predicted abnormalities and the provided guideline context. Do NOT invent findings not supported by either.

2. IMPRESSION: Provide a concise clinical conclusion based strictly on the Findings.

Use formal radiology phrasing. If a predicted abnormality has low probability, use hedging language ("possible", "cannot be excluded", "suggestive of"). If the predicted labels indicate a normal study, state that clearly.

=== USER PROMPT ===
PREDICTED ABNORMALITIES (from vision model, with probabilitie

## 10. Ablation Harness

The problem statement asks for ablations to quantify the contribution of retrieval, staging, and verification. Here's a minimal harness — run each variant on the same set of test images and record the verifier outputs.

Variants to compare:

1. **`no_rag`** — feed predicted labels directly to the LLM with no retrieved context.
2. **`rag_definition_only`** — retrieve only `type == 'definition'` snippets.
3. **`rag_full`** — full hybrid retrieval (our system).
4. **`rag_full + verifier_loop`** — if verifier flags inconsistency, regenerate once.

Once you have real generation (not dryrun), aggregate over a held-out test set and report **label-F1** + **hallucination rate** + **consistency rate** per variant. That's your headline ablation table.

In [21]:
def run_no_rag(predicted_labels: Dict[str, float], generator: Generator) -> dict:
    report = generator.generate(predicted_labels, snippets=[])
    return {
        "predicted_labels":   predicted_labels,
        "retrieved_snippets": [],
        "report":             report,
        "verifier":           verify(report, predicted_labels),
    }

def run_definition_only(predicted_labels: Dict[str, float],
                        retriever: Retriever, generator: Generator) -> dict:
    snippets = retriever.retrieve(predicted_labels)
    snippets = [s for s in snippets if s["type"] == "definition"]
    report = generator.generate(predicted_labels, snippets)
    return {
        "predicted_labels":   predicted_labels,
        "retrieved_snippets": snippets,
        "report":             report,
        "verifier":           verify(report, predicted_labels),
    }

def ablation_summary(results: List[dict]) -> dict:
    n = len(results)
    if n == 0:
        return {}
    consistent = sum(1 for r in results if r["verifier"]["is_consistent"])
    halluc = sum(len(r["verifier"]["hallucinated"]) for r in results)
    missed = sum(len(r["verifier"]["missed"])      for r in results)
    return {
        "n_examples":         n,
        "consistency_rate":   consistent / n,
        "avg_hallucinations": halluc / n,
        "avg_missed":         missed / n,
    }

# Tiny example with the mock prediction:
variants = {
    "no_rag":       run_no_rag(mock_predicted_labels, generator),
    "rag_def_only": run_definition_only(mock_predicted_labels, retriever, generator),
    "rag_full":     run_pipeline(mock_predicted_labels, retriever, generator),
}

print("Variant outputs (verifier subset shown):")
for name, r in variants.items():
    print(f"\n--- {name} ---")
    print(f"  retrieved snippets: {len(r['retrieved_snippets'])}")
    print(f"  verifier: {r['verifier']}")

Variant outputs (verifier subset shown):

--- no_rag ---
  retrieved snippets: 0
  verifier: {'mentioned': ['Effusion', 'Infiltration'], 'predicted_confident': ['Effusion', 'Infiltration'], 'hallucinated': [], 'missed': [], 'is_consistent': True}

--- rag_def_only ---
  retrieved snippets: 2
  verifier: {'mentioned': ['Consolidation', 'Effusion', 'Infiltration'], 'predicted_confident': ['Effusion', 'Infiltration'], 'hallucinated': ['Consolidation'], 'missed': [], 'is_consistent': False}

--- rag_full ---
  retrieved snippets: 8
  verifier: {'mentioned': ['Atelectasis', 'Consolidation', 'Effusion', 'Infiltration', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax'], 'predicted_confident': ['Effusion', 'Infiltration'], 'hallucinated': ['Atelectasis', 'Consolidation', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax'], 'missed': [], 'is_consistent': False}
